# Introduction
This notebook follows the workflow described below:
1. Run PGM Power Flow (PF), and use the PF results as the true values.
2. Generate measurements based on the standard deviations given by the user.
3. Run PGM State Estimation (SE).
4. Compare the true state, measurements, and the estimated state.

# The Network
The network used in this file is
[CIGRE MV Network](https://pandapower.readthedocs.io/en/latest/networks/cigre.html#medium-voltage-distribution-network), with 15 buses(nodes) and 15 lines.

In [1]:
import os
import sys

# add the parent directory to the path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
os.makedirs("../results/cigre_mv_play", exist_ok=True)

In [2]:
from modules.h_matrix_gen import HMatrixGenerator
from power_grid_model.validation import assert_valid_input_data
from power_grid_model import (
    CalculationMethod,
    CalculationType,
    ComponentType,
    PowerGridModel
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
from modules.se_input_maker import SEInputMaker

# Setting up the experiment 
Set the parameters you want here.

In [ ]:
u_sigma = 10  # for all voltage sensors
u_rated_general = 20000
u_angle_sigma = u_sigma / u_rated_general * 2 * np.pi
sensor_setting = 'from_i'
i_sigma = 2  # for all line current sensors
i_rated_general = 145
i_angle_sigma = i_sigma / i_rated_general * 2 * np.pi
general_seed = 42
np.random.seed(general_seed)

# choose which nodes to put voltage sensors on
voltage_sensor_id = [0, 1, 6, 12]

# choose the status of the switches
# change this line to change the switch status. 1: closed, 0: open
switch_status = [1, 1, 1]
switch_index = [14, 12, 13]  # S1,S2,S3, line 29, 27, 28

In [5]:
se_input_maker = SEInputMaker(
    network_name="cigre_mv",
    u_sigma=u_sigma,
    sensor_setting=sensor_setting,
    u_rated_general=u_rated_general,
    voltage_sensor_id=voltage_sensor_id,
    switch_status=switch_status,
    switch_index=switch_index,
    i_sigma=i_sigma,
    i_rated_general=i_rated_general,
    general_seed=42,
)

# Import and Run Power Flow

## Validation the Power Flow Input Data

In [6]:
input_data = se_input_maker.input_data

In [7]:
assert_valid_input_data(
    input_data, calculation_type=CalculationType.power_flow, symmetric=True
)

## Run PF

In [8]:
model = PowerGridModel(input_data)

output_data = model.calculate_power_flow(
    symmetric=True,
    error_tolerance=1e-8,
    max_iterations=20,
    calculation_method=CalculationMethod.newton_raphson,
)

In [9]:
output_data = se_input_maker.run_pf()

In [10]:
# Node result
print("------node result------")
print(pd.DataFrame(output_data[ComponentType.node]))

------node result------
    id  energized      u_pu              u   u_angle             p  \
0    0          1  1.025966  112856.229179 -0.008165  4.496799e+07   
1    1          1  0.991582   19831.644969 -0.636045 -1.983900e+07   
2    2          1  0.978106   19562.113365 -0.644199  5.581224e-09   
3    3          1  0.956581   19131.627175 -0.657057 -5.017000e+05   
4    4          1  0.954733   19094.650651 -0.658399 -4.316500e+05   
5    5          1  0.953933   19078.658249 -0.659061 -7.275000e+05   
6    6          1  0.953680   19073.599235 -0.659026 -5.480500e+05   
7    7          1  0.953868   19077.365132 -0.658802 -7.650000e+04   
8    8          1  0.955445   19108.893586 -0.657067 -5.868500e+05   
9    9          1  0.954767   19095.348778 -0.657626 -5.737500e+05   
10  10          1  0.954218   19084.355392 -0.658471 -5.433000e+05   
11  11          1  0.954312   19086.232386 -0.658549 -3.298000e+05   
12  12          1  0.993182   19863.632624 -0.638858 -2.001000e+07

In [11]:
# Line result
print("------line result------")
print(pd.DataFrame(output_data[ComponentType.line]))

------line result------
    id  energized   loading        p_from        q_from     i_from  \
0   15          1  0.509082  2.298216e+06  1.019260e+06  73.191855   
1   16          1  0.515913  2.275315e+06  1.038494e+06  73.816859   
2   17          1  0.336549  1.514882e+06  5.551592e+05  48.688949   
3   18          1  0.166089  7.613785e+05  2.241935e+05  23.998501   
4   19          1  0.017607  3.339202e+04  5.085860e+04   1.841144   
5   20          1  0.125273 -5.912577e+05 -1.032858e+05  18.164572   
6   21          1  0.245727  1.128226e+06  3.376846e+05  35.582001   
7   22          1  0.115524  5.538665e+05 -1.321962e+04  16.750992   
8   23          1  0.033788  1.024187e+04 -1.616207e+05   4.899246   
9   24          1  0.081607  2.220418e+05  3.005989e+05  11.277868   
10  25          1  0.419007  2.751379e+06  5.704057e+05  81.671224   
11  26          1  0.413416  2.667452e+06  5.195041e+05  80.596083   
12  27          1  0.108165 -5.146691e+05 -5.989626e+04  15.683972

# Build the Measurement Matrix H

In [12]:
h_matrix_gen = HMatrixGenerator(input_data=input_data)
(
    H_voltage,
    H_line_current_from,
    H_line_current_to,
    H_transformer_current_from,
    H_transformer_current_to,
) = h_matrix_gen.generate_h_matrix(whole_return=False)
H_line_current = np.concatenate(
    (H_line_current_from, H_line_current_to), axis=0)
H_transformer_current = np.concatenate(
    (H_transformer_current_from, H_transformer_current_to), axis=0
)
h_matrix_auto = np.concatenate(
    (H_voltage, H_line_current_from, H_transformer_current_from), axis=0
)

# Correctness check of the H Matrix

In [13]:
# extract the result from PGM Power Flow
x_pf = np.array(output_data[ComponentType.node]["u"]) * np.exp(
    1j * np.array(output_data[ComponentType.node]["u_angle"])
)
x_pf_abs = abs(x_pf)
x_pf_angle = np.angle(x_pf)
I_from_pf = np.array(output_data[ComponentType.line]["i_from"])
I_transformer_from_pf = np.array(
    output_data[ComponentType.transformer]["i_from"])
I_transformer_to_pf = np.array(output_data[ComponentType.transformer]["i_to"])
# put all the
result_pf = np.concatenate(
    (x_pf_abs, I_from_pf, I_transformer_from_pf), axis=0)
result_pf_abs = abs(result_pf)

Hx = h_matrix_auto @ x_pf
Hx_abs = abs(Hx)

In [14]:
# Create a DataFrame with both Hx and z side by side
diff_in_percent = np.divide(
    Hx_abs - result_pf_abs, result_pf_abs, where=result_pf_abs != 0
)
ratios = np.divide(
    Hx_abs, result_pf_abs, out=np.ones_like(Hx_abs), where=result_pf_abs != 0
)
comparison_df = pd.DataFrame(
    {
        "Hx_abs": Hx_abs,
        "result_pf_abs": result_pf_abs,
        "diff in percent": diff_in_percent,  # 0 is the ideal value.
        "ratio": ratios,  # 1 is the ideal value.
    }
)
print(comparison_df)

           Hx_abs  result_pf_abs  diff in percent  ratio
0   112856.229179  112856.229179     0.000000e+00    1.0
1    19831.644969   19831.644969     0.000000e+00    1.0
2    19562.113365   19562.113365     0.000000e+00    1.0
3    19131.627175   19131.627175     0.000000e+00    1.0
4    19094.650651   19094.650651     0.000000e+00    1.0
5    19078.658249   19078.658249     0.000000e+00    1.0
6    19073.599235   19073.599235     0.000000e+00    1.0
7    19077.365132   19077.365132     0.000000e+00    1.0
8    19108.893586   19108.893586     0.000000e+00    1.0
9    19095.348778   19095.348778     0.000000e+00    1.0
10   19084.355392   19084.355392     0.000000e+00    1.0
11   19086.232386   19086.232386     0.000000e+00    1.0
12   19863.632624   19863.632624     0.000000e+00    1.0
13   19467.318835   19467.318835     0.000000e+00    1.0
14   19229.378935   19229.378935     0.000000e+00    1.0
15      73.191855      73.191855     2.329908e-15    1.0
16      73.816859      73.81685

/tmp/ipykernel_1331029/2479924429.py:2: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  diff_in_percent = np.divide(


In [15]:
# check how many ratios are with 1- 1e-6 and 1+ 1e-6
print(
    f"Percentage of ratios with 1- 1e-6 and 1+ 1e-6: {np.sum(np.abs(ratios - 1) < 1e-6) / len(ratios) * 100}%"
)
print(
    f"Percentage of ratios with 1- 1e-6 and 1+ 1e-6: {np.sum(np.abs(ratios - 1) < 1e-6) / len(ratios) * 100}%"
)

# check how many ratios are with 1- 1e-3 and 1+ 1e-3
print(
    f"Percentage of ratios with 1- 1e-3 and 1+ 1e-3: {np.sum(np.abs(ratios - 1) < 1e-3) / len(ratios) * 100}%"
)
print(
    f"Percentage of ratios with 1- 1e-3 and 1+ 1e-3: {np.sum(np.abs(ratios - 1) < 1e-3) / len(ratios) * 100}%"
)

Percentage of ratios with 1- 1e-6 and 1+ 1e-6: 100.0%
Percentage of ratios with 1- 1e-6 and 1+ 1e-6: 100.0%
Percentage of ratios with 1- 1e-3 and 1+ 1e-3: 100.0%
Percentage of ratios with 1- 1e-3 and 1+ 1e-3: 100.0%


# Compute current angle
Since PGM PF and PGM SE does not provide the current angle directly, we need to calculate it with the H Matrix, and the voltage given by PGM PF.

In [16]:
i_pf = H_line_current_from @ x_pf
i_pf_angle = np.angle(i_pf)
i_pf_abs = abs(i_pf)

# PGM State Estimation

## Setting up the Sensors

In [17]:
input_data_se, sym_voltage_sensor, sym_power_sensor, sym_current_sensor = se_input_maker.setup_se_input()

## Check if all the current sensors were correctly configured

In [18]:
i_measured_abs = sym_current_sensor["i_measured"]
i_measured_angle = sym_current_sensor["i_angle_measured"]

In [19]:
compare_i_pf_abs_i_measured = pd.DataFrame(
    {
        "i_pf_abs": i_pf_abs,
        "i_measured": i_measured_abs,
        "diff in percent": np.divide(
            i_pf_abs - i_measured_abs, i_measured_abs, where=i_measured_abs != 0
        ),
        "ratio": np.divide(
            i_pf_abs,
            i_measured_abs,
            out=np.ones_like(i_pf_abs),
            where=i_measured_abs != 0,
        ),
    }
)
print(compare_i_pf_abs_i_measured)

     i_pf_abs  i_measured  diff in percent     ratio
0   73.191855   74.681997        -0.019953  0.980047
1   73.816859   73.402066         0.005651  1.005651
2   48.688949   50.632014        -0.038376  0.961624
3   23.998501   28.567590        -0.159940  0.840060
4    1.841144    1.138684         0.616905  1.616905
5   18.164572   17.462162         0.040225  1.040225
6   35.582001   40.319639        -0.117502  0.882498
7   16.750992   19.053296        -0.120835  0.879165
8    4.899246    3.490823         0.403465  1.403465
9   11.277868   12.905548        -0.126123  0.873877
10  81.671224   80.280971         0.017317  1.017317
11  80.596083   79.198894         0.017642  1.017642
12  15.683972   16.409859        -0.044235  0.955765
13  12.063742    6.323902         0.907642  1.907642
14  63.406190   58.231437         0.088865  1.088865


/tmp/ipykernel_1331029/3163086731.py:5: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  "diff in percent": np.divide(


In [20]:
compare_i_pf_angle_i_measured = pd.DataFrame(
    {
        "i_pf_angle": i_pf_angle,
        "i_measured_angle": i_measured_angle,
        "diff in percent": np.divide(
            i_pf_angle - i_measured_angle, i_measured_angle, where=i_measured_angle != 0
        ),
        "ratio": np.divide(
            i_pf_angle,
            i_measured_angle,
            out=np.ones_like(i_pf_angle),
            where=i_measured_angle != 0,
        ),
    }
)
print(compare_i_pf_angle_i_measured)

    i_pf_angle  i_measured_angle  diff in percent     ratio
0    -1.053481         -1.126577        -0.064883  0.935117
1    -1.072377         -1.204042        -0.109352  0.890648
2    -1.008329         -0.967478         0.042224  1.042224
3    -0.944764         -1.062804        -0.111065  0.888935
4    -1.648880         -1.832475        -0.100190  0.899810
5     2.309847          2.500377        -0.076200  0.923800
6    -0.947887         -0.977237        -0.030034  0.969966
7    -0.633762         -0.624984         0.014046  1.014046
8     0.849041          0.663828         0.279007  1.279007
9    -1.591647         -1.662415        -0.042569  0.957431
10   -0.843279         -0.828859         0.017397  1.017397
11   -0.840243         -0.989869        -0.151157  0.848843
12    2.366709          2.415549        -0.020219  0.979781
13    1.841735          1.763654         0.044272  1.044272
14   -0.769372         -0.807292        -0.046971  0.953029


/tmp/ipykernel_1331029/4086249027.py:5: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  "diff in percent": np.divide(


## Validation of SE Input Data

In [21]:
assert_valid_input_data(
    input_data=input_data_se,
    calculation_type=CalculationType.state_estimation,
    symmetric=True,
)
model_se = PowerGridModel(input_data_se)

## Run SE

In [22]:
output_data_se = model_se.calculate_state_estimation(
    symmetric=True,
    error_tolerance=1e-8,
    max_iterations=20,
    calculation_method=CalculationMethod.iterative_linear,
)

In [23]:
# Line result
print("------line result------")
print(pd.DataFrame(output_data_se[ComponentType.line]))

------line result------
    id  energized   loading        p_from        q_from     i_from  \
0   15          1  0.473310  2.001687e+06  1.193034e+06  67.841558   
1   16          1  0.481907  1.981952e+06  1.216797e+06  68.629992   
2   17          1  0.365617  1.649179e+06  5.963412e+05  52.905104   
3   18          1  0.187583  8.328586e+05  3.306567e+05  27.090049   
4   19          1  0.005689 -2.286865e+04 -1.483394e+04   0.824887   
5   20          1  0.132331 -6.295501e+05 -7.691585e+04  19.188014   
6   21          1  0.263987  1.214230e+06  3.574118e+05  38.230459   
7   22          1  0.122115  5.793546e+05 -8.659661e+04  17.706669   
8   23          1  0.027273  1.566308e+04 -1.298243e+05   3.954595   
9   24          1  0.082482  1.936168e+05  3.233480e+05  11.369809   
10  25          1  0.419812  2.755829e+06  5.856412e+05  81.827339   
11  26          1  0.411663  2.540545e+06  9.355674e+05  80.237812   
12  27          1  0.114870 -5.481040e+05 -5.056476e+04  16.656133

In [24]:
# Node result
print("------node result------")
print(pd.DataFrame(output_data_se[ComponentType.node]))

------node result------
    id  energized      u_pu              u   u_angle             p  \
0    0          1  1.026011  112861.196320 -0.008900  4.431125e+07   
1    1          1  0.991555   19831.098115 -0.636749 -2.012939e+07   
2    2          1  0.978235   19564.690903 -0.642726 -8.332305e-09   
3    3          1  0.956894   19137.871623 -0.652044 -1.072905e+05   
4    4          1  0.954891   19097.826045 -0.653512 -6.363669e+05   
5    5          1  0.953928   19078.550934 -0.654170 -8.551071e+05   
6    6          1  0.953978   19079.563805 -0.654104 -5.252343e+05   
7    7          1  0.954173   19083.455261 -0.653861 -8.134604e+04   
8    8          1  0.955750   19115.006117 -0.651941 -2.630183e+05   
9    9          1  0.955025   19100.508336 -0.652544 -6.341719e+05   
10  10          1  0.954556   19091.122998 -0.653505 -5.633298e+05   
11  11          1  0.954628   19092.558812 -0.653573 -1.929988e+05   
12  12          1  0.993928   19878.550896 -0.636442 -1.935693e+07